In [ ]:
"""
Main script to run hierarchical PPO (HDRL) experiments for two related
stochastic lot-sizing problems:

  1) S-CLSP
     Stochastic Capacitated Lot-Sizing Problem with product-level setup costs and
     (optionally) setup times in a *sequence-independent* setting.

  2) S-CLSP-SD
     Sequence-Dependent variant where setup times/costs depend on the transition
     between consecutive products, represented by full matrices (st_sd, k_sd).

How to use this script
----------------------
- Select the exact benchmark instance as reported in the paper by setting:
      INSTANCE_ID = "x.y"    (e.g., "1.1", "2.7", "3.4")
  The script automatically maps the paper ID to the corresponding problem variant
  (S-CLSP or S-CLSP-SD) and builds the full instance configuration.

- Optionally choose the training protocol by setting:
      VARIANT = "comparison" or "optimized"

  * "comparison" reproduces the baseline training configuration used for
    direct benchmarking against reference single-agent PPO results (aligned rollout
    horizon/updates and evaluation protocol).

  * "optimized" uses the hierarchical training configuration tailored for HDRL
    (larger rollouts per update and the optimized schedule/settings reported in
    the paper), intended to maximize performance and stability.

Pipeline overview
-----------------
- The script constructs a hyperparameter/config dictionary (HYP) from the selected
  INSTANCE_ID, creates the matching environment via `make_env_from_hyp` (S-CLSP) or
  `make_env_from_hyp2` (S-CLSP-SD), trains a hierarchical PPO agent (`PPO_Hier` /
  `PPO_Hier2`), and finally evaluates the best checkpoint over multiple Monte Carlo
  runs using greedy (deterministic) action selection.

Outputs
-------
- Training logs printed to console (and optionally to Weights & Biases if enabled).
- Model checkpoints saved under `HYP["out_dir"]`:
    * high_joint_best.pt
    * low_joint_best.pt
  plus any periodic checkpoints the trainer produces.
- Final evaluation summary printed at the end (mean and std over test runs).

Reproducibility
---------------
The script sets seeds and deterministic PyTorch flags to make results as
reproducible as possible across runs, subject to GPU nondeterminism limitations.
"""
# ============================================================
# 0) Global selector
# ============================================================
# Set the instance ID exactly as in the paper (e.g., "1.1", "2.1", "3.1").
# Optionally choose VARIANT:
#   - "comparison" aligns training horizon/updates with the baseline setup
#   - "optimized"  uses the optimized hierarchical configuration
INSTANCE_ID = "1.1"
VARIANT = "optimized"   # {"comparison", "optimized"}


# ====================
# 1) Shared imports 
# ====================
import os
import random
import numpy as np
import torch


# ============================================================
# 2) Shared reproducibility utilities
# ============================================================
def set_global_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"

    np.random.seed(seed)
    random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# 3) Instance builders (S-CLSP)
# ============================================================
def build_base_hyp(K):
    """
    Build the base hyperparameter dictionary (HYP) used across experiments.

    This function creates and returns a configuration dictionary containing:
      (i) core problem size information (number of products K), and
      (ii) common PPO and environment-related hyperparameters shared by all runs.

    The following CLSP instance-specific fields are intentionally initialized to
    ``None`` and must be filled by downstream builders (e.g., ``build_hyp_set1``
    or ``build_hyp_set2_case_K4``):
      - capacity_factor, mu, bs, h, b, k, st, cov

    :param K: Number of products in the instance.
    :type K: int

    :return: Base hyperparameter dictionary. It includes PPO training settings
             (e.g., horizon, updates, gamma, GAE lambda, clipping ratio),
             learning-rate schedule parameters, logging/output settings, and
             placeholders for instance-specific CLSP parameters to be set by
             higher-level builders.
    :rtype: dict
    """
    HYP = {

        # =====================================================
        # (A) Problem definition — CLSP instance parameters
        # =====================================================
        "K": K,                   # number of products
        "capacity_factor": None,  # capacity scaling factor
        "mu": None,               # mean demand vector
        "bs": None,               # batch size per product
        "h": None,                # holding cost
        "b": None,                # backorder cost
        "k": None,                # setup cost
        "st": None,               # setup times
        "cov": None,              # demand coefficient of variation


        # =====================================================
        # (B) Environment / rollout configuration
        # =====================================================
        "horizon": 1024,          # episode length
        "steps_per_update": 1024, # rollout size per PPO update


        # =====================================================
        # (C) PPO core hyperparameters
        # =====================================================
        "updates": 6000,          # total PPO updates
        "gamma": 0.99,            # discount factor
        "lam": 0.95,              # GAE parameter
        "clip_ratio": 0.2,        # PPO clipping parameter
        "train_iters": 10,        # gradient steps per update
        "batch_size": 256,        # minibatch size
        "entropy_coef": 1e-2,     # entropy regularization
        "kl_target": 0.03,        # KL divergence threshold


        # =====================================================
        # (D) Learning-rate schedule (stepwise decay)
        # =====================================================
        "lr_high": 1e-4,          # initial LR (high-level policy)
        "lr_low":  1e-4,          # initial LR (low-level policy)
        "lr_high_levels": [1e-4, 5e-5, 3e-5],
        "lr_low_levels":  [1e-4, 5e-5, 3e-5],
        "lr_drop_patience_evals": 7,  # non-improving eval tolerance


        # =====================================================
        # (E) Logging, evaluation and early stopping
        # =====================================================
        "print_every": 10,
        "out_dir": ".",
        "use_wandb": False,
        "eval_seed_base": 10000,
        "early_stop_patience": 15,
    }

    return HYP


def build_hyp_set1(K, cov_level, cf):
    """
    Build the hyperparameter dictionary (HYP) for SET 1
    — Evaluation Case 1: Scalability under homogeneous demand.

    SET 1 is designed to evaluate how the hierarchical PPO policy
    scales as the number of products (K) increases, under a fully
    symmetric and controlled cost-demand structure.

    Instance characteristics:
      - Homogeneous demand: μ_i = 4  
      - Homogeneous backorder cost: b_i = 9  
      - Fixed time-between-orders: TBO_i = 10  
      - No setup times: s_i = 0
      - Demand variability:
            * cov = 0.14  ("low")
            * cov = 0.58  ("high")
      - Batch size identical across products

    :param K: Number of products (scalability dimension).
    :type K: int

    :param cov_level: Demand variability level ("low" or "high").
    :type cov_level: str

    :param cf: Capacity factor (1.1 or 1.5).
    :type cf: float

    :return: Fully specified hyperparameter dictionary for SET 1.
    :rtype: dict
    """
    # -----------------------------
    # Basic validation of inputs
    # -----------------------------
    assert cov_level in ("low", "high")
    assert cf in (1.1, 1.5)

    # Initialize base PPO + structural configuration
    HYP = build_base_hyp(K)

    # -----------------------------
    # Homogeneous structural values
    # -----------------------------
    MU = 4.0      # mean demand
    H = 1.0       # holding cost
    B = 9.0       # backorder cost
    TBO = 10.0    # time-between-orders parameter

    # Even batch size (kept identical across products)
    bs_val = (K // 2) * 2 # or 5 or 10 for scalability cases

    # -----------------------------
    # Demand variability selection
    # -----------------------------
    if cov_level == "low":
        COV = 0.14
    else:
        COV = 0.58

    # -----------------------------
    # Populate CLSP instance fields
    # -----------------------------
    HYP["capacity_factor"] = cf
    HYP["mu"]  = np.full(K, MU, dtype=np.float32)
    HYP["bs"]  = np.full(K, bs_val, dtype=int)
    HYP["h"]   = np.full(K, H, dtype=np.float32)
    HYP["b"]   = np.full(K, B, dtype=np.float32)
    HYP["cov"] = np.full(K, COV, dtype=np.float32)

    # Setup cost derived from classical EOQ-style relation:
    #   k_i = h_i * TBO_i^2 * μ_i / 2
    TBO_vec = np.full(K, TBO, dtype=np.float32)
    HYP["k"] = H * (TBO_vec ** 2) * HYP["mu"] / 2.0

    # No setup times in SET 1
    HYP["st"] = np.zeros(K, dtype=int)

    return HYP


def build_hyp_set2_case_K4(case_id):
    """
    Build the hyperparameter dictionary (HYP) for SET 2
    — Evaluation Case 2, Subcase 2.1:
      Heterogeneous demand with asymmetric cost structures (K = 4).

    SET 2 evaluates the hierarchical PPO policy under fixed problem size
    but heterogeneous economic structure. Demand levels differ across
    products and cost parameters vary according to predefined benchmark
    cases (12–18).

    Base structure common to all cases:
      - K = 4 (fixed dimension)
      - μ = [2, 2, 4, 4]  (two low-demand, two high-demand products)
      - cov = 0.58  (high variability)
      - capacity_factor = 1.1
      - bs_i = 4  
      - h_i = 1   
      - s_i = 0   (no setup times)

    The selected case_id defines the backorder cost vector b_i and
    the time-between-orders vector TBO_i.

    :param case_id: One of {12, 13, 14, 15, 16, 17, 18}.
    :type case_id: int
    :return: Fully specified hyperparameter dictionary for SET 2.
    :rtype: dict
    """

    # -----------------------------
    # Basic validation
    # -----------------------------
    assert case_id in (12, 13, 14, 15, 16, 17, 18)

    K = 4
    HYP = build_base_hyp(K)

    # =====================================================
    # (1) Common structural configuration (all cases)
    # =====================================================
    MU_vec = np.array([2.0, 2.0, 4.0, 4.0], dtype=np.float32)
    H = 1.0
    COV = 0.58
    cf = 1.1
    bs_val = (K // 2) * 2  # = 4

    HYP["capacity_factor"] = cf
    HYP["mu"]  = MU_vec.copy()
    HYP["bs"]  = np.full(K, bs_val, dtype=int)
    HYP["h"]   = np.full(K, H, dtype=np.float32)
    HYP["cov"] = np.full(K, COV, dtype=np.float32)
    HYP["st"]  = np.zeros(K, dtype=int)  # no setup times in SET 2

    # =====================================================
    # (2) Case-specific economic structure
    #     (b_i and TBO_i patterns)
    # =====================================================
    if case_id == 12:
        # Homogeneous low cost / low TBO
        TBO_vec = np.full(K, 5.0, dtype=np.float32)
        b_vec   = np.full(K, 9.0, dtype=np.float32)

    elif case_id == 13:
        # Homogeneous low cost / high TBO
        TBO_vec = np.full(K, 10.0, dtype=np.float32)
        b_vec   = np.full(K, 9.0, dtype=np.float32)

    elif case_id == 14:
        # Homogeneous high cost / low TBO
        TBO_vec = np.full(K, 5.0, dtype=np.float32)
        b_vec   = np.full(K, 49.0, dtype=np.float32)

    elif case_id == 15:
        # Homogeneous high cost / high TBO
        TBO_vec = np.full(K, 10.0, dtype=np.float32)
        b_vec   = np.full(K, 49.0, dtype=np.float32)

    elif case_id == 16:
        # Demand-aligned asymmetry:
        # μ=2 → (b=9,  TBO=5)
        # μ=4 → (b=49, TBO=10)
        TBO_vec = np.array([5.0, 5.0, 10.0, 10.0], dtype=np.float32)
        b_vec   = np.array([9.0, 9.0, 49.0, 49.0], dtype=np.float32)

    elif case_id == 17:
        # Inverted asymmetry:
        # μ=2 → (b=49, TBO=10)
        # μ=4 → (b=9,  TBO=5)
        TBO_vec = np.array([10.0, 10.0, 5.0, 5.0], dtype=np.float32)
        b_vec   = np.array([49.0, 49.0, 9.0, 9.0], dtype=np.float32)

    else:  # case_id == 18
        # Fully mixed structure (heterogeneous across all dimensions)
        TBO_vec = np.array([5.0, 10.0, 10.0, 5.0], dtype=np.float32)
        b_vec   = np.array([9.0, 49.0, 9.0, 49.0], dtype=np.float32)

    # =====================================================
    # (3) Derived setup cost
    # =====================================================
    HYP["b"] = b_vec

    # EOQ-style relation:
    #   k_i = h_i * TBO_i^2 * μ_i / 2
    HYP["k"] = H * (TBO_vec ** 2) * HYP["mu"] / 2.0

    return HYP


def build_hyp_set3_demand_case_K10(demand_case_id):
    """
    Build the hyperparameter dictionary (HYP) for SET 3
    — Evaluation Case 2, Subcase 2.2:
      Structured demand patterns (synthetic scenarios).

    SET 3 is designed to test robustness of the hierarchical PPO policy under
    non-homogeneous demand structures that go beyond the benchmark instances.
    In addition to heterogeneous μ-vectors, some cases inject correlation
    metadata that the environment can use to generate *structured* demands
    (overriding the default independent per-product sampling).

    Demand scenarios (demand_case_id)
    --------------------------------
      1) Two-block demand:        5×μ=2, 5×μ=8
      2) Three-block demand:      4×μ=2, 3×μ=4, 3×μ=8
      3) Stepped demand:          3×μ=2, 3×μ=3, μ=[6,7,8,9]
      4) Paired demand (+1):      μ=[2,2,4,4,6,6,8,8,10,10]  + pair_plus_one metadata
      5) Paired demand (same):    μ=[2,2,4,4,6,6,8,8,10,10]  + pair_same metadata
      6) Grouped demand (same):   μ=[2,2,2,2, 4,4,4, 8,8,8]  + group_same metadata
      7) Increasing demand:       μ=[2,3,4,5,6,7,8,9,10,11]

    Base structure (common across SET 3)
    -----------------------------------
      - K = 10 (fixed)
      - cov = 0.58 for all products
      - capacity_factor = 1.1
      - h_i = 1, b_i = 9, st_i = 0
      - k_i derived from a fixed TBO = 10:
            k_i = h_i * TBO^2 * μ_i / 2

    :param demand_case_id: Demand scenario identifier in {1, ..., 7}.
    :type demand_case_id: int
    :return: Fully specified hyperparameter dictionary for SET 3.
    :rtype: dict
    """
    # -----------------------------
    # Basic validation
    # -----------------------------
    assert demand_case_id in (1, 2, 3, 4, 5, 6, 7)

    # =====================================================
    # (1) Fixed SET 3 size + base PPO configuration
    # =====================================================
    K = 10
    HYP = build_base_hyp(K)

    # =====================================================
    # (2) Common structural parameters (all cases)
    # =====================================================
    H = 1.0
    B = 9.0
    TBO = 10.0
    COV = 0.58
    cf = 1.1
    bs_val = 10

    # =====================================================
    # (3) Case-specific demand structure (μ vectors)
    #     + optional correlation metadata for the environment
    # =====================================================
    if demand_case_id == 1:
        # Two-block demand: 5 low, 5 high
        MU_vec = np.array([2] * 5 + [8] * 5, dtype=np.float32)

    elif demand_case_id == 2:
        # Three-block demand: low / medium / high
        MU_vec = np.array([2] * 4 + [4] * 3 + [8] * 3, dtype=np.float32)

    elif demand_case_id == 3:
        # Stepped / mixed demand levels
        MU_vec = np.array([2] * 3 + [3] * 3 + [6, 7, 8, 9], dtype=np.float32)

    elif demand_case_id == 4:
        # Paired demand (+1): enforce structured pairwise dependence
        MU_vec = np.array([2, 2, 4, 4, 6, 6, 8, 8, 10, 10], dtype=np.float32)
        HYP["demand_corr_mode"] = "pair_plus_one"
        HYP["demand_corr_pairs"] = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    elif demand_case_id == 5:
        # Paired demand (same): enforce identical sampled demand within each pair
        MU_vec = np.array([2, 2, 4, 4, 6, 6, 8, 8, 10, 10], dtype=np.float32)
        HYP["demand_corr_mode"] = "pair_same"
        HYP["demand_corr_pairs"] = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    elif demand_case_id == 6:
        # Grouped demand (same): shared sampled demand within each group
        MU_vec = np.array([2] * 4 + [4] * 3 + [8] * 3, dtype=np.float32)
        HYP["demand_corr_mode"] = "group_same"
        HYP["demand_corr_groups"] = [
            [0, 1, 2, 3],
            [4, 5, 6],
            [7, 8, 9],
        ]

    else:  # demand_case_id == 7
        # Strictly increasing demand across products
        MU_vec = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype=np.float32)

    # =====================================================
    # (4) Populate CLSP instance fields
    # =====================================================
    HYP["capacity_factor"] = cf
    HYP["mu"] = MU_vec.copy()
    HYP["bs"] = np.full(K, bs_val, dtype=int)
    HYP["h"] = np.full(K, H, dtype=np.float32)
    HYP["b"] = np.full(K, B, dtype=np.float32)
    HYP["cov"] = np.full(K, COV, dtype=np.float32)
    HYP["st"] = np.zeros(K, dtype=int)

    # =====================================================
    # (5) Derived setup cost from fixed TBO
    # =====================================================
    TBO_vec = np.full(K, TBO, dtype=np.float32)
    HYP["k"] = H * (TBO_vec ** 2) * HYP["mu"] / 2.0

    return HYP


# ============================================================
# 4) Instance builders (S-CLSP-SD)
# ============================================================
def build_base_hyp_sd(K):
    """
    Build the base hyperparameter dictionary (HYP) for S-CLSP-SD experiments.

    This function creates a configuration dictionary containing:
      (i) core problem size information (K), and
      (ii) common PPO and environment-related hyperparameters shared by SD runs.

    The following instance-specific fields are initialized to ``None`` and
    must be filled by downstream builders (e.g., ``build_hyp_eval_case``):
      - capacity_factor, mu, bs, h, b, cov
      - st_sd, k_sd  (sequence-dependent setup matrices)

    Legacy / compatibility fields are also included:
      - st (vector, non-SD setup times)
      - k  (vector, non-SD setup costs)

    :param K: Number of products in the instance (fixed to 10 in the SD suite).
    :type K: int

    :return: Base hyperparameter dictionary for SD experiments.
    :rtype: dict
    """
    HYP = {
        "K": K,

        # =====================================================
        # (A) Problem definition — S-CLSP economic parameters
        # =====================================================
        "capacity_factor": None,
        "mu": None,
        "bs": None,
        "h": None,
        "b": None,
        "cov": None,

        # =====================================================
        # (B) SD structures — sequence-dependent setups
        # =====================================================
        "st_sd": None,   # setup-time matrix (K×K, float, diag = 0)
        "k_sd": None,    # setup-cost matrix (K×K, int,   diag = 0)

        # =====================================================
        # (C) Legacy / fallback vectors (compatibility)
        # =====================================================
        "st": None,
        "k": None,

        # =====================================================
        # (D) Environment / rollout configuration
        # =====================================================
        "horizon": 1024,
        "steps_per_update": 1024,

        # =====================================================
        # (E) PPO core hyperparameters
        # =====================================================
        "updates": 6000,
        "gamma": 0.99,
        "lam": 0.95,
        "clip_ratio": 0.2,
        "train_iters": 10,
        "batch_size": 256,
        "entropy_coef": 1e-2,
        "kl_target": 0.03,

        # =====================================================
        # (F) Learning-rate schedule (stepwise decay)
        # =====================================================
        "lr_high": 1e-4,
        "lr_low":  1e-4,
        "lr_high_levels": [1e-4, 5e-5, 3e-5],
        "lr_low_levels":  [1e-4, 5e-5, 3e-5],
        "lr_drop_patience_evals": 7,

        # =====================================================
        # (G) Logging, evaluation and early stopping
        # =====================================================
        "print_every": 10,
        "out_dir": ".",
        "use_wandb": False,
        "eval_seed_base": 10000,
        "early_stop_patience": 15,
    }
    return HYP


def compute_CAP(mu, bs_val, cf):
    """
    Compute the effective production capacity (CAP) for a given instance.

    In both S-CLSP and S-CLSP-SD, capacity is scaled proportionally to
    total expected demand. The standard formulation used here is:

        CAP = ceil( cf * ( sum(mu) / bs ) )

    :param mu: Mean demand vector (μ_i for each product).
    :type mu: numpy.ndarray

    :param bs_val: Batch size (assumed identical across products in SD suite).
    :type bs_val: int

    :param cf: Capacity factor (capacity tightness multiplier).
    :type cf: float

    :return: Integer production capacity (in batches per period).
    :rtype: int
    """
    return int(np.ceil(cf * (mu.sum() / bs_val)))


def _st_float(sf, CAP, K):
    """
    Convert a dimensionless setup-time scaling factor into an
    absolute setup time (in batches).

    In the SD setting, setup times are generated from relative
    structure parameters (sf_ij). These are mapped into actual
    time units using:

        st_ij = sf * CAP / K

    Intuition:
        - CAP / K acts as a normalization term relative to
          per-product average capacity.
        - sf controls the relative magnitude of setup times
          with respect to available capacity.
        - If sf <= 0, setup time is set to 0 (no transition penalty).

    This function is used internally when constructing the
    sequence-dependent setup-time matrix st_sd.

    :param sf: Setup scaling factor (dimensionless).
    :type sf: float

    :param CAP: Total production capacity (batches per period).
    :type CAP: int

    :param K: Number of products.
    :type K: int

    :return: Setup time value (float, typically rounded later).
    :rtype: float
    """
    if sf <= 0:
        return 0.0
    return sf * CAP / K


def build_st_sd_matrix(sd_structure_id, K, CAP, sf_global, seed=123):
    """
    Build the sequence-dependent setup-time matrix (st_sd) for S-CLSP-SD.

    The returned matrix has shape (K × K), where:
        - st[i, j] represents the setup time incurred when switching
          production from product i to product j.
        - Diagonal elements are zero (no setup when continuing
          with the same product).
        - Values are expressed in batches (float, rounded to 1 decimal).

    Three structural patterns are supported:

        1) Two-family structure
           - Products are divided into two equal families.
           - Low setup times within the same family.
           - Higher setup times across families.

        2) Two critical products
           - Two designated products (crit1, crit2) generate
             disproportionately high setup times when involved
             in a transition.
           - Remaining transitions have moderate base levels.

        3) Fully heterogeneous
           - Randomized transition intensities drawn from a
             uniform distribution.
           - Produces a dense and irregular setup structure.

    Setup times are scaled proportionally to system capacity using:
        st_ij = round( _st_float(sf_ij, CAP, K), 1 )

    :param sd_structure_id: Identifier of the SD structure {1, 2, 3}.
    :type sd_structure_id: int

    :param K: Number of products.
    :type K: int

    :param CAP: Production capacity (in batches per period).
    :type CAP: int

    :param sf_global: Global setup scaling factor (controls magnitude).
    :type sf_global: float

    :param seed: Random seed used only for heterogeneous case (id=3).
    :type seed: int

    :return: Setup-time matrix st_sd (K × K, float32, diag=0).
    :rtype: numpy.ndarray
    """
    assert sd_structure_id in (1, 2, 3)
    st = np.zeros((K, K), dtype=np.float32)

    # =====================================================
    # (1) Two-family structure
    # =====================================================
    if sd_structure_id == 1:
        # Split products into two equal families
        fam1 = set(range(0, K // 2))
        fam2 = set(range(K // 2, K))

        # Normalize scaling relative to reference SF = 0.20
        s = sf_global / 0.20

        # Relative transition intensities
        sf11 = 0.30 * s   # within family 1
        sf22 = 0.30 * s   # within family 2
        sf12 = 0.80 * s   # across families

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue

                if (i in fam1) and (j in fam1):
                    sf_ij = sf11
                elif (i in fam2) and (j in fam2):
                    sf_ij = sf22
                else:
                    sf_ij = sf12

                st[i, j] = round(_st_float(sf_ij, CAP, K), 1)

        np.fill_diagonal(st, 0.0)
        return st

    # =====================================================
    # (2) Two critical products
    # =====================================================
    if sd_structure_id == 2:
        # Two designated "critical" products
        crit1, crit2 = 1, 7

        s = sf_global / 0.20

        # Relative transition intensities
        sf_base  = 0.35 * s   # default transitions
        sf_to_c1 = 0.95 * s   # transitions involving crit1
        sf_to_c2 = 0.75 * s   # transitions involving crit2
        sf_c1c2  = 1.00 * s   # transition between crit1 and crit2

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue

                if (i == crit1 and j == crit2) or (i == crit2 and j == crit1):
                    sf_ij = sf_c1c2
                elif (i == crit1) or (j == crit1):
                    sf_ij = sf_to_c1
                elif (i == crit2) or (j == crit2):
                    sf_ij = sf_to_c2
                else:
                    sf_ij = sf_base

                st[i, j] = round(_st_float(sf_ij, CAP, K), 1)

        np.fill_diagonal(st, 0.0)
        return st

    # =====================================================
    # (3) Fully heterogeneous structure
    # =====================================================
    rng = np.random.default_rng(seed)

    # Draw relative intensities from a uniform distribution
    sf_mat = rng.uniform(0.15, 0.60, size=(K, K)).astype(np.float32)
    np.fill_diagonal(sf_mat, 0.0)

    s = sf_global / 0.20

    for i in range(K):
        for j in range(K):
            st[i, j] = round(_st_float(sf_mat[i, j] * s, CAP, K), 1)

    np.fill_diagonal(st, 0.0)
    return st


def _clip_int(x, lo, hi):
    """
    Round to nearest integer and clip to [lo, hi].

    :param x: Input value.
    :type x: float
    :param lo: Lower bound (inclusive).
    :type lo: int
    :param hi: Upper bound (inclusive).
    :type hi: int

    :return: Clipped integer value.
    :rtype: int
    """
    return int(np.clip(np.rint(x), lo, hi))


def build_k_sd_matrix(sd_structure_id, K, setup_cost_max=200, seed=123):
    """
    Build the sequence-dependent setup-cost matrix (k_sd) for S-CLSP-SD.

    Produces a (K × K) integer matrix where k_sd[i, j] is the setup cost of
    switching from product i to product j (diag = 0). Three patterns are supported:
      1) two families, 2) two critical products, 3) fully heterogeneous.

    :param sd_structure_id: SD structure identifier {1, 2, 3}.
    :type sd_structure_id: int
    :param K: Number of products.
    :type K: int
    :param setup_cost_max: Maximum allowed setup cost (values clipped to this bound).
    :type setup_cost_max: int
    :param seed: RNG seed (affects randomized costs).
    :type seed: int

    :return: Setup-cost matrix k_sd (K × K, int32, diag=0).
    :rtype: numpy.ndarray
    """
    assert sd_structure_id in (1, 2, 3)
    rng = np.random.default_rng(seed)
    ksd = np.zeros((K, K), dtype=np.int32)

    # (1) Two-family structure: low within family, high across families
    if sd_structure_id == 1:
        fam1 = set(range(0, K // 2))
        fam2 = set(range(K // 2, K))

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                if (i in fam1) and (j in fam1):
                    base = 60.0 + rng.normal(0, 4.0)
                elif (i in fam2) and (j in fam2):
                    base = 140.0 + rng.normal(0, 6.0)
                else:
                    base = 190.0 + rng.normal(0, 5.0)
                ksd[i, j] = _clip_int(base, 0, setup_cost_max)

        np.fill_diagonal(ksd, 0)
        return ksd

    # (2) Two critical products: very high between criticals, high when involving them
    if sd_structure_id == 2:
        crit1, crit2 = 1, 7

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                if (i == crit1 and j == crit2) or (i == crit2 and j == crit1):
                    base = setup_cost_max + rng.normal(0, 2.0)
                elif (i == crit1) or (j == crit1):
                    base = 190.0 + rng.normal(0, 5.0)
                elif (i == crit2) or (j == crit2):
                    base = 150.0 + rng.normal(0, 7.0)
                else:
                    base = 80.0 + rng.normal(0, 6.0)
                ksd[i, j] = _clip_int(base, 0, setup_cost_max)

        np.fill_diagonal(ksd, 0)
        return ksd

    # (3) Fully heterogeneous: random costs in [20, setup_cost_max]
    raw = rng.uniform(20.0, setup_cost_max, size=(K, K))
    np.fill_diagonal(raw, 0.0)

    for i in range(K):
        for j in range(K):
            ksd[i, j] = _clip_int(raw[i, j], 0, setup_cost_max)

    np.fill_diagonal(ksd, 0)
    return ksd


def build_mu_eval_case(eval_case_id, K=10):
    """
    Return the mean-demand vector (mu) for one of the 7 SD evaluation cases (K=10).

    Cases:
      1,2,4,5 -> homogeneous mu=4
      3       -> two-family demand (2 then 8)
      6       -> two critical products with mu={2,8} and rest 4
      7       -> strictly increasing mu=[2..11]

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int
    :param K: Number of products (must be 10 for these cases).
    :type K: int

    :return: Mean demand vector mu (shape: (K,), float32).
    :rtype: numpy.ndarray
    """
    assert K == 10

    if eval_case_id in (1, 2, 4, 5):
        return np.full(K, 4.0, dtype=np.float32)

    if eval_case_id == 3:
        return np.array([2.0] * 5 + [8.0] * 5, dtype=np.float32)

    if eval_case_id == 6:
        mu = np.full(K, 4.0, dtype=np.float32)
        crit1, crit2 = 1, 7
        mu[crit1] = 2.0
        mu[crit2] = 8.0
        return mu

    if eval_case_id == 7:
        return np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype=np.float32)

    raise ValueError("eval_case_id must be in {1..7}")


def build_cov_eval_case(eval_case_id, K=10):
    """
    Return the demand variability vector (cov) for a given SD evaluation case.

    By design:
      - Low variability is used only for cases 1 and 4.
      - High variability is used for all other cases.

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int
    :param K: Number of products (defaults to 10 for the SD suite).
    :type K: int

    :return: Coefficient-of-variation vector (shape: (K,), float32).
    :rtype: numpy.ndarray
    """
    # Demand variability levels used by the SD suite
    COV_LOW = 0.14
    COV_HIGH = 0.58
    cov_val = COV_LOW if eval_case_id in (1, 4) else COV_HIGH
    return np.full(K, cov_val, dtype=np.float32)


def sd_structure_id_from_eval_case(eval_case_id):
    """
    Map an evaluation case id to the SD setup structure id used to build (st_sd, k_sd).

    Mapping:
      - cases {1,2,3} -> 1 (two-family structure)
      - cases {4,5,6} -> 2 (two critical products)
      - case  {7}     -> 3 (fully heterogeneous)

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int

    :return: SD structure identifier in {1, 2, 3}.
    :rtype: int
    """
    if eval_case_id in (1, 2, 3):
        return 1
    if eval_case_id in (4, 5, 6):
        return 2
    if eval_case_id == 7:
        return 3
    raise ValueError("eval_case_id must be in {1..7}")


def build_hyp_eval_case(eval_case_id):
    """
    Build the full hyperparameter dictionary (HYP) for the selected SD evaluation case.

    This function assembles:
      - Core CLSP parameters (mu, cov, h, b, bs, capacity_factor),
      - Derived capacity CAP,
      - Sequence-dependent setup matrices (st_sd, k_sd),

    Notes:
      - The SD suite is defined for K=10 in this benchmark-style setup.
      - Legacy fields (st, k) are also populated for compatibility with shared code.

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int

    :return: Fully specified HYP dictionary for the chosen SD instance.
    :rtype: dict
    """
    assert eval_case_id in (1, 2, 3, 4, 5, 6, 7)

    # -----------------------
    # 0) Global configuration
    # -----------------------
    # Seed for SD instance generation (st_sd and k_sd matrices)
    INSTANCE_SEED = 123

    # ============================================================
    # 2) Common scales / knobs (shared across the 7 SD cases)
    # ============================================================
    # Problem size (fixed for SD evaluation suite)
    K = 10

    # Capacity factor (capacity is derived later from cf * sum(mu) / bs)
    CF = 1.1

    # SD setup-time scaling factor (controls magnitude of st_sd entries)
    SF = 0.20

    # Upper bound for generated setup costs in k_sd
    SETUP_COST_MAX = 200

    # Economic parameters (aligned with the S-CLSP baselines)
    H_HOLD = 1.0      # holding cost
    B_BACK = 9.0      # backorder cost
    BS_VAL = 10       # batch size (constant across products for SD suite)

    # Legacy reference TBO used to compute fallback k (non-SD vector)
    # kept to avoid breaking code paths expecting HYP["k"]
    TBO = 10.0

    # Base PPO/env config + placeholders
    HYP = build_base_hyp_sd(K)

    # Case-dependent demand parameters
    mu_vec = build_mu_eval_case(eval_case_id, K)
    cov_vec = build_cov_eval_case(eval_case_id, K)

    # Core economic structure
    HYP["capacity_factor"] = CF
    HYP["mu"] = mu_vec
    HYP["bs"] = np.full(K, BS_VAL, dtype=np.int32)
    HYP["h"] = np.full(K, H_HOLD, dtype=np.float32)
    HYP["b"] = np.full(K, B_BACK, dtype=np.float32)
    HYP["cov"] = cov_vec

    # Derived capacity (in batches per period)
    CAP = compute_CAP(mu_vec, BS_VAL, CF)
    HYP["CAP"] = CAP

    # SD setup structures
    sd_id = sd_structure_id_from_eval_case(eval_case_id)
    HYP["st_sd"] = build_st_sd_matrix(sd_id, K, CAP, SF, seed=INSTANCE_SEED)
    HYP["k_sd"] = build_k_sd_matrix(sd_id, K, setup_cost_max=SETUP_COST_MAX, seed=INSTANCE_SEED)

    # Legacy / fallback vectors (kept for compatibility with shared components)
    HYP["st"] = np.zeros(K, dtype=np.int32)
    HYP["k"] = H_HOLD * (TBO ** 2) * HYP["mu"] / 2.0

    return HYP


def print_case_summary(eval_case_id, HYP):
    mu = HYP["mu"]
    CAP = HYP.get("CAP", None)
    bs = int(HYP["bs"][0])
    cov = HYP["cov"]

    print("\n" + "=" * 80)
    print(f"EVAL CASE {eval_case_id}")
    print(f"mu: {mu.astype(int)} | sum(mu)={mu.sum():.1f} | CAP={CAP} | bs={bs} | cf={HYP['capacity_factor']}")
    print(f"cov: {cov[0]:.3f} (all equal) | CAP/K = {CAP / HYP['K']:.3f}")
    print("-" * 80)
    print("Setup-times matrix st_sd (float, 1 decimal, diag=0) [batches]:")
    print(HYP["st_sd"])
    print("\nSetup-costs matrix k_sd (int, diag=0, <= setup_cost_max):")
    print(HYP["k_sd"])
    print("=" * 80 + "\n")


# ============================================================
# 5) Paper instance registry (ID -> builder parameters)
# ============================================================
# Case I: homogeneous demand, K in {7..15}, cf in {1.1, 1.5}, cov in {low, high}
CASE_I_REGISTRY = {
    # -------------------------
    # Low demand variability
    # -------------------------
    "1.1":  dict(K=7,  cf=1.1, cov="low"),
    "1.2":  dict(K=7,  cf=1.5, cov="low"),
    "1.3":  dict(K=8,  cf=1.1, cov="low"),
    "1.4":  dict(K=8,  cf=1.5, cov="low"),
    "1.5":  dict(K=9,  cf=1.1, cov="low"),
    "1.6":  dict(K=9,  cf=1.5, cov="low"),
    "1.7":  dict(K=10, cf=1.1, cov="low"),
    "1.8":  dict(K=10, cf=1.5, cov="low"),

    "1.17": dict(K=11, cf=1.1, cov="low"),
    "1.18": dict(K=11, cf=1.5, cov="low"),
    "1.19": dict(K=12, cf=1.1, cov="low"),
    "1.20": dict(K=12, cf=1.5, cov="low"),
    "1.21": dict(K=13, cf=1.1, cov="low"),
    "1.22": dict(K=13, cf=1.5, cov="low"),
    "1.23": dict(K=14, cf=1.1, cov="low"),
    "1.24": dict(K=14, cf=1.5, cov="low"),
    "1.25": dict(K=15, cf=1.1, cov="low"),
    "1.26": dict(K=15, cf=1.5, cov="low"),

    # -------------------------
    # High demand variability
    # -------------------------
    "1.9":  dict(K=7,  cf=1.1, cov="high"),
    "1.10": dict(K=7,  cf=1.5, cov="high"),
    "1.11": dict(K=8,  cf=1.1, cov="high"),
    "1.12": dict(K=8,  cf=1.5, cov="high"),
    "1.13": dict(K=9,  cf=1.1, cov="high"),
    "1.14": dict(K=9,  cf=1.5, cov="high"),
    "1.15": dict(K=10, cf=1.1, cov="high"),
    "1.16": dict(K=10, cf=1.5, cov="high"),

    "1.27": dict(K=11, cf=1.1, cov="high"),
    "1.28": dict(K=11, cf=1.5, cov="high"),
    "1.29": dict(K=12, cf=1.1, cov="high"),
    "1.30": dict(K=12, cf=1.5, cov="high"),
    "1.31": dict(K=13, cf=1.1, cov="high"),
    "1.32": dict(K=13, cf=1.5, cov="high"),
    "1.33": dict(K=14, cf=1.1, cov="high"),
    "1.34": dict(K=14, cf=1.5, cov="high"),
    "1.35": dict(K=15, cf=1.1, cov="high"),
    "1.36": dict(K=15, cf=1.5, cov="high"),
}

# Case II, subset 1 (K=4): paper IDs 2.1..2.7 map to case_id 12..18
CASE_II_K4_REGISTRY = {
    "2.1": dict(case_id=12),
    "2.2": dict(case_id=13),
    "2.3": dict(case_id=14),
    "2.4": dict(case_id=15),
    "2.5": dict(case_id=16),
    "2.6": dict(case_id=17),
    "2.7": dict(case_id=18),
}

# Case II, subset 2 (K=10 patterns): paper IDs 2.8..2.14 map to demand_case_id 1..7
CASE_II_K10_REGISTRY = {
    "2.8":  dict(demand_case_id=1),
    "2.9":  dict(demand_case_id=2),
    "2.10": dict(demand_case_id=3),
    "2.11": dict(demand_case_id=4),
    "2.12": dict(demand_case_id=5),
    "2.13": dict(demand_case_id=6),
    "2.14": dict(demand_case_id=7),
}

# Case III (SD): paper IDs 3.1..3.7 map to eval_case_id 1..7
CASE_III_SD_REGISTRY = {
    "3.1": dict(eval_case_id=1),
    "3.2": dict(eval_case_id=2),
    "3.3": dict(eval_case_id=3),
    "3.4": dict(eval_case_id=4),
    "3.5": dict(eval_case_id=5),
    "3.6": dict(eval_case_id=6),
    "3.7": dict(eval_case_id=7),
}


def build_hyp_from_instance_id(instance_id, variant="optimized"):
    instance_id = instance_id.strip()

    # ============================================================
    # Case I (S-CLSP) — homogeneous demand suite
    # ============================================================
    if instance_id in CASE_I_REGISTRY:
        cfg = CASE_I_REGISTRY[instance_id]
        HYP = build_hyp_set1(cfg["K"], cfg["cov"], cfg["cf"])
        problem = "SCLSP"

        # Apply fair-comparison vs optimized training protocol
        if variant == "comparison":
            HYP["horizon"] = 256
            HYP["steps_per_update"] = 256
            HYP["updates"] = 10_000
        elif variant == "optimized":
            HYP["horizon"] = 1024
            HYP["steps_per_update"] = 1024
            HYP["updates"] = 6000
        else:
            raise ValueError("variant must be {'comparison','optimized'}")

        exp_name = f"paper_{instance_id}_{variant}_K{cfg['K']}_cov{cfg['cov']}_cf{cfg['cf']}"
        return problem, HYP, exp_name

    # ============================================================
    # Case II (S-CLSP) — heterogeneous K=4 suite
    # ============================================================
    if instance_id in CASE_II_K4_REGISTRY:
        cfg = CASE_II_K4_REGISTRY[instance_id]
        HYP = build_hyp_set2_case_K4(cfg["case_id"])
        problem = "SCLSP"

        if variant not in ("comparison", "optimized"):
            raise ValueError("variant must be {'comparison','optimized'}")

        exp_name = f"paper_{instance_id}_{variant}_case{cfg['case_id']}"
        return problem, HYP, exp_name

    # ============================================================
    # Case II (S-CLSP) — structured K=10 demand patterns
    # ============================================================
    if instance_id in CASE_II_K10_REGISTRY:
        cfg = CASE_II_K10_REGISTRY[instance_id]
        HYP = build_hyp_set3_demand_case_K10(cfg["demand_case_id"])
        problem = "SCLSP"

        if variant not in ("comparison", "optimized"):
            raise ValueError("variant must be {'comparison','optimized'}")

        exp_name = f"paper_{instance_id}_{variant}_dcase{cfg['demand_case_id']}"
        return problem, HYP, exp_name

    # ============================================================
    # Case III (S-CLSP-SD) — sequence-dependent suite
    # ============================================================
    if instance_id in CASE_III_SD_REGISTRY:
        cfg = CASE_III_SD_REGISTRY[instance_id]
        HYP = build_hyp_eval_case(cfg["eval_case_id"])
        problem = "SCLSP_SD"

        if variant not in ("comparison", "optimized"):
            raise ValueError("variant must be {'comparison','optimized'}")

        exp_name = f"paper_{instance_id}_{variant}_SDcase{cfg['eval_case_id']}"
        return problem, HYP, exp_name

    raise ValueError(f"Unknown INSTANCE_ID '{instance_id}'. Not found in paper registry.")


# ============================================================
# 6) MAIN 1: S-CLSP
# ============================================================
def run_sclsp_with_hyp(HYP, exp_name, SEED=123):
    """
    Execute hierarchical PPO experiments for the S-CLSP (Stochastic
    Capacitated Lot-Sizing Problem) without sequence-dependent setups.

    This routine encapsulates the full experimental pipeline for the
    sequence-independent setting:
        1) Reproducibility configuration (seed + deterministic flags).
        2) Selection and construction of a predefined benchmark instance.
        3) Hyperparameter dictionary (HYP) assembly.
        4) Environment creation via ``make_env_from_hyp``.
        5) Hierarchical PPO training (``PPO_Hier``).
        6) Final evaluation using the best saved model checkpoints.

    The experiment family is selected through ``EXP_SET``:
        - 1 → Scalability analysis under homogeneous demand (variable K).
        - 2 → Heterogeneous demand with asymmetric cost structures (K = 4).
        - 3 → Structured demand patterns (K = 10, synthetic scenarios).

    Each family has an associated configuration dictionary defining
    the structural parameters of the instance (e.g., K, demand pattern,
    cost structure, variability level).

    Notes
    -----
    - The S-CLSP setting assumes setup costs are product-specific
      but independent of the previously produced product.
    - All instance-specific parameters are injected into a single
      hyperparameter dictionary (HYP), which is then passed to both
      the environment and the trainer for full reproducibility.
    - Checkpoints and logs are written to ``HYP["out_dir"]``.

    :return: None
    :rtype: None
    """
    # -----------------------
    # 0) Global configuration
    # -----------------------
    # Fixed random seed for reproducibility of:
    #   - demand sampling
    #   - network initialization
    #   - rollout trajectories
    set_global_seeds(SEED)

    # Automatically select GPU if available
    DEVICE = get_device()

    # Optional experiment tracking (activated via HYP["use_wandb"])
    import wandb

    # Environment factory and hierarchical PPO implementation
    from CLSP_HDRL import make_env_from_hyp, PPO_Hier

    # ============================================================
    # 1) WandB (optional)
    # ============================================================
    if HYP.get("use_wandb", False):
        wandb.login()
        run = wandb.init(project="HDRL_CLSP", config=HYP, name=exp_name)
    else:
        run = None

    # ============================================================
    # 2) Environment and agent initialization
    # ============================================================
    env = make_env_from_hyp(HYP, seed=SEED)
    trainer = PPO_Hier(env, HYP, device=DEVICE, seed=SEED)

    # ============================================================
    # 3) Training
    # ============================================================
    trainer.train()
    if run is not None:
        run.finish()

    # ============================================================
    # 4) Final evaluation
    # ============================================================
    res = trainer.test(
        n_runs=10,
        warmup=1000,
        greedy=True,
        ckpt_high=os.path.join(HYP["out_dir"], "high_joint_best.pt"),
        ckpt_low=os.path.join(HYP["out_dir"], "low_joint_best.pt"),
        env_overrides={"horizon": 10_000},
    )

    print("==============================================")
    print(f"Experiment: {exp_name}")
    print("Mean cost over runs:", res["mean_of_means"])
    print("Std. of mean costs: ", res["std_of_means"])
    print("==============================================")


# ============================================================
# 4) MAIN 2: S-CLSP-SD
# ============================================================
def run_sclsp_sd_with_hyp(HYP, exp_name, SEED=123):
    """
    Execute hierarchical PPO experiments for the S-CLSP-SD (Stochastic CLSP with
    Sequence-Dependent setup times and setup costs).

    This routine encapsulates the full experimental pipeline for the SD setting:
        1) Reproducibility configuration (run seed + deterministic flags).
        2) Selection of one of 7 predefined SD evaluation instances (K = 10).
        3) Construction of the hyperparameter dictionary (HYP), including:
            - standard S-CLSP economic parameters (mu, h, b, cov, bs, capacity_factor),
            - sequence-dependent setup structures:
                  * st_sd: setup-time matrix (float, shape K×K, diag = 0)
                  * k_sd : setup-cost matrix (int,   shape K×K, diag = 0)
            - optional legacy vectors (st, k) kept for compatibility.
        4) Environment creation via ``make_env_from_hyp2`` (SD-aware env).
        5) Hierarchical PPO training using ``PPO_Hier2`` (SD-aware algorithm wiring).
        6) Final evaluation using the best saved checkpoints.

    The SD instances are indexed by ``EVAL_CASE_ID ∈ {1..7}`` and cover three
    structural families:
        - Cases 1–3: two-family transition structure
        - Cases 4–6: two critical products transition structure
        - Case 7:    fully heterogeneous transition structure

    Notes
    -----
    - ``SEED`` controls run-level randomness (policy initialization, rollouts).
    - ``INSTANCE_SEED`` controls instance generation for SD matrices (st_sd, k_sd).
      Keeping it fixed ensures that the SD structure is identical across reruns.
    - Checkpoints and logs are written to ``HYP["out_dir"]``.

    :return: None
    :rtype: None
    """
    # -----------------------
    # 0) Global configuration
    # -----------------------
    # Seed for run reproducibility (net init, rollouts, demand sampling, etc.)
    set_global_seeds(SEED)
    DEVICE = get_device()

    # SD-aware environment factory and hierarchical PPO implementation
    from CLSP_HDRL import make_env_from_hyp2, PPO_Hier2
    
    # ============================================================
    # Construct HYP + run training + final test
    # ============================================================
    # WandB optional
    if HYP.get("use_wandb", False):
        import wandb
        wandb.login()
        run = wandb.init(project="HDRL_CLSP", config=HYP, name=exp_name)
    else:
        run = None

    env = make_env_from_hyp2(HYP, seed=SEED)
    trainer = PPO_Hier2(env, HYP, device=DEVICE, seed=SEED)

    trainer.train()
    if run is not None:
        run.finish()

    res = trainer.test(
        n_runs=10,
        warmup=1000,
        greedy=True,
        ckpt_high=os.path.join(HYP["out_dir"], "high_joint_best.pt"),
        ckpt_low=os.path.join(HYP["out_dir"], "low_joint_best.pt"),
        env_overrides={"horizon": 10_000},
    )

    print("==============================================")
    print(f"Experiment: {exp_name}")
    print("Mean cost over runs:", res["mean_of_means"])
    print("Std. of mean costs: ", res["std_of_means"])
    print("==============================================")


# ============================================================
# 7) Entry point
# ============================================================
# Build HYP from paper instance ID
PROBLEM, HYP, exp_name = build_hyp_from_instance_id(INSTANCE_ID, VARIANT)


if PROBLEM == "SCLSP":
    run_sclsp_with_hyp(HYP, exp_name, SEED=123)
elif PROBLEM == "SCLSP_SD":
    # Optional: print SD summary (requires eval_case_id; parse from registry)
    eval_case_id = CASE_III_SD_REGISTRY.get(INSTANCE_ID, {}).get("eval_case_id", None)
    if eval_case_id is not None:
        print_case_summary(eval_case_id, HYP)
    run_sclsp_sd_with_hyp(HYP, exp_name, SEED=123)
else:
    raise ValueError('PROBLEM must be "SCLSP" or "SCLSP_SD".')